## 1. Load Required Libraries and GTFS Data

This cell imports all necessary libraries (pandas, osmnx, networkx) and loads the GTFS transit data files (routes, stops, trips, stop_times, shapes) along with the OpenStreetMap graph for walking route calculations.

In [1]:
import os
import math
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from pathlib import Path

# Configure paths
GTFS_DIR = Path(r"gtfsAlex")
GRAPH_XML = Path(r"new.osm")

# Load GTFS
routes = pd.read_csv(GTFS_DIR / "routes.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")

# Ensure correct dtypes
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")

print(f"Loaded routes={len(routes)}, stops={len(stops)}, trips={len(trips)}, stop_times={len(stop_times)}")

# Load OSM graph (walkable)
g = ox.graph_from_xml(filepath=str(GRAPH_XML), bidirectional=True)
# Simplify graph for routing consistency
g = ox.convert.to_undirected(g)
print(f"Graph loaded with {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


Loaded routes=104, stops=441, trips=192, stop_times=2548
Graph loaded with 45785 nodes, 65856 edges


## 2. Build Trip-to-Stop Mappings

This cell processes the GTFS data to create essential mappings:
- Trip start and end stops (based on stop_sequence)
- Trip-to-route relationships
- Trip-to-shape relationships for visualization
- Complete stop sets for each trip

In [2]:
# Build trip -> ordered stops, and canonical start/end stop per trip
# We assume stop_times stop_sequence ascending defines direction along a trip.

# Filter trips that have stop_times data
trips_with_stops = trips[trips['trip_id'].isin(stop_times['trip_id'])]
print(f"Trips with stop_times data: {len(trips_with_stops)}")

# Build ordered stops for each trip
ordered_stops = (
    stop_times
    .sort_values(['trip_id','stop_sequence'])
    .merge(trips_with_stops[['trip_id','route_id']], on='trip_id', how='left')
)

# Pick start and end stop_id per trip
start_stop_per_trip = ordered_stops.groupby('trip_id').first()['stop_id']
end_stop_per_trip = ordered_stops.groupby('trip_id').last()['stop_id']

print(f"Trips with start/end stops: {len(start_stop_per_trip)}")

# Create convenience dicts
trip_to_start_stop = start_stop_per_trip.to_dict()
trip_to_end_stop = end_stop_per_trip.to_dict()

# Also collect all trip->set(stops) for proximity to the whole trip corridor
trip_to_all_stops = (
    ordered_stops.groupby('trip_id')['stop_id'].apply(lambda s: set(s.values)).to_dict()
)

# Create trip->route mapping for reference
trip_to_route = trips.set_index('trip_id')['route_id'].to_dict()

# Create trip->shape_id mapping for visualization
trip_to_shape = trips.set_index('trip_id')['shape_id'].to_dict()

# Build shape_id -> list of (lat, lon) points for visualization
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")
shape_to_points = {}
for shape_id, group in shapes.groupby('shape_id'):
    points = group.sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
    shape_to_points[shape_id] = points

print(f"Built {len(shape_to_points)} shape geometries for visualization")


Trips with stop_times data: 192
Trips with start/end stops: 192
Built 192 shape geometries for visualization


## 3. Map GTFS Stops to Graph Nodes

This cell maps each GTFS stop to its nearest node in the OpenStreetMap walking graph. This is essential for calculating walking paths between transit stops.

In [3]:
# Map each GTFS stop to nearest graph node
# Prepare stop_id -> (lat, lon) -> nearest node
stop_coords = stops.set_index('stop_id')[['stop_lat','stop_lon']]

# Vectorized nearest nodes using OSMnx
xs = stop_coords['stop_lon'].values
ys = stop_coords['stop_lat'].values
nearest_nodes = ox.distance.nearest_nodes(g, X=xs, Y=ys)

stop_to_node = {stop_id: node for stop_id, node in zip(stop_coords.index.values, nearest_nodes)}
print(f"Mapped {len(stop_to_node)} stops to nearest graph nodes")

# Convenience: node positions
node_x = nx.get_node_attributes(g, 'x')
node_y = nx.get_node_attributes(g, 'y')


Mapped 441 stops to nearest graph nodes


## 4. Find Proximity Candidates

This cell identifies potential walking connections between trips by finding:
1. Trip starts that are near other trips' corridor stops (excluding same trip's start)
2. Trip ends that are near other trips' corridor stops

Uses haversine distance filtering with a 300m radius for initial candidate screening.

In [4]:
# Compute proximity candidates:
# For each trip t1, find trips t2 whose start node is near any stop node of t1,
# and trips t3 whose end node is near any stop node of t1.
# We'll use a radius (meters) on great-circle distance to get candidates; exact walking
# shortest path will be computed in the next step.
# IMPORTANT: We ignore t1's own first stop when considering proximity to other trip starts.

from math import radians, sin, cos, asin, sqrt

def haversine_m(lat1, lon1, lat2, lon2):
    # returns meters
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Build stop_id -> (lat, lon)
stop_lat = stops.set_index('stop_id')['stop_lat'].to_dict()
stop_lon = stops.set_index('stop_id')['stop_lon'].to_dict()

# Precompute start/end node for every trip
trip_to_start_node = {t: stop_to_node[s] for t, s in trip_to_start_stop.items() if s in stop_to_node}
trip_to_end_node = {t: stop_to_node[s] for t, s in trip_to_end_stop.items() if s in stop_to_node}

# Parameters
NEAR_RADIUS_M = 300  # coarse proximity filter in meters

# Build a fast lookup of trip t -> list of nodes for all its stops
trip_to_nodes = {t: [stop_to_node[s] for s in stops_set if s in stop_to_node]
                  for t, stops_set in trip_to_all_stops.items()}

# For reverse lookup: node -> (lat, lon)
node_latlon = {n: (node_y.get(n), node_x.get(n)) for n in g.nodes()}

# Candidate list: (start_trip_id, end_trip_id, start_node, end_node)
# Two types:
# 1) t2 start near t1 corridor: (t2 -> t1)
# 2) t3 end near t1 corridor: (t3 -> t1)
# Note: We ignore t1's own first stop when considering proximity to other trip starts.
proximity_candidates = []

trip_ids = list(trip_to_all_stops.keys())


for t1 in trip_ids:
    t1_nodes = trip_to_nodes.get(t1, [])
    if not t1_nodes:
        continue

    # compute t1's start node (to be excluded when checking other starts)
    t1_start_stop = trip_to_start_stop.get(t1)
    t1_start_node = stop_to_node.get(t1_start_stop) if t1_start_stop in stop_to_node else None

    # 1) t2 starts near t1
    for t2, t2_start_node in trip_to_start_node.items():
        if t2 == t1:
            continue
        # skip if the closest t1 node would be exactly its own start node
        lat2, lon2 = node_latlon.get(t2_start_node, (None, None))
        if lat2 is None:
            continue
        # coarse: check nearest t1 node by haversine radius
        near = False
        closest_t1_node = None
        closest_dist = float('inf')
        for n in t1_nodes:
            if t1_start_node is not None and n == t1_start_node:
                # ignore t1's start node when comparing to other starts
                continue
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat2, lon2)
            if d < closest_dist:
                closest_dist = d
                closest_t1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near and closest_t1_node is not None:
            # Corrected: Creates pathway from t1 -> t2
            proximity_candidates.append((t1, t2, closest_t1_node, t2_start_node))

    # 2) t3 ends near t1 (unchanged - no exclusion needed)
    for t3, t3_end_node in trip_to_end_node.items():
        if t3 == t1:
            continue
        lat3, lon3 = node_latlon.get(t3_end_node, (None, None))
        if lat3 is None:
            continue
        near = False
        closest_t1_node = None
        closest_dist = float('inf')
        for n in t1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat3, lon3)
            if d < closest_dist:
                closest_dist = d
                closest_t1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            proximity_candidates.append((t3, t1, t3_end_node, closest_t1_node))

print(f"Proximity candidates: {len(proximity_candidates)}")


Proximity candidates: 6506


## 6. Calculate Walking Pathways with Enhanced Data

This cell processes proximity candidates to calculate actual walking paths using NetworkX shortest path algorithm. 

**Enhanced features:**
- Computes exact walking distances using OSM edge lengths
- Resolves precise start/end stop IDs for each pathway
- Adds stop sequences within trips
- Includes stop coordinates (lat/lon)
- Creates comprehensive pathway records with route information

In [5]:
# Shortest walking paths for candidates and assemble DataFrame
# We find the closest actual stop node on t1 to the candidate endpoint node as the "end_stop_id".
# Path cost uses edge length in meters if available, else fallback to 1.

# Build reverse node->stop_id index to resolve end_stop_id
node_to_stop_ids = {}
for sid, n in stop_to_node.items():
    node_to_stop_ids.setdefault(n, []).append(sid)

# Helper to pick the nearest t1 stop node to a given node by straight distance
from math import inf

def nearest_node_in_set(target_node, node_set):
    ty, tx = node_y.get(target_node), node_x.get(target_node)
    if ty is None:
        return None
    best_node, best_d = None, inf
    for n in node_set:
        ny, nx_ = node_y.get(n), node_x.get(n)
        if ny is None:
            continue
        d = haversine_m(ty, tx, ny, nx_)
        if d < best_d:
            best_d = d
            best_node = n
    return best_node

# Ensure we have edge lengths (meters)
if not nx.get_edge_attributes(g, 'length'):
    g = ox.distance.add_edge_lengths(g)

# 1. Prerequisites: Lookup Dictionaries
# Map Trip -> Set of Stops (for validity check)
trip_to_stop_set = stop_times.groupby('trip_id')['stop_id'].apply(set).to_dict()

# Map Trip -> Route ID (Needed to populate the route_id columns)
trip_to_route = trips.set_index('trip_id')['route_id'].to_dict()

# Map Route ID -> Agency ID (Needed to populate the agency_id columns)
route_to_agency = routes.set_index('route_id')['agency_id'].to_dict()

# Map Route ID -> Route Name (Needed to populate the route_name columns)
route_to_name = routes.set_index('route_id')['route_long_name'].to_dict()

# Map Route ID -> Route Short Name (Needed to populate the route_short_name columns)
route_to_short_name = routes.set_index('route_id')['route_short_name'].to_dict()

# Map Trip ID -> Trip Headsign (Needed to populate the trip_headsign columns)
trip_to_headsign = trips.set_index('trip_id')['trip_headsign'].to_dict()

# Map (trip_id, stop_id) -> stop_sequence for sequence lookup
trip_stop_to_sequence = stop_times.set_index(['trip_id', 'stop_id'])['stop_sequence'].to_dict()

# Map stop_id -> coordinates for coordinate lookup
stop_to_coords = stops.set_index('stop_id')[['stop_lat', 'stop_lon']].to_dict('index')

records = []
failed = 0

print(f"Processing {len(proximity_candidates)} proximity candidates...")

for start_trip_id, end_trip_id, start_node, coarse_end_node in proximity_candidates:
    
    # [1] Refine end node: find the nearest node used by the specific end_trip
    # Get all nodes associated with the end trip
    target_trip_nodes = [stop_to_node[s] for s in trip_to_stop_set.get(end_trip_id, []) if s in stop_to_node]
    
    if not target_trip_nodes:
        continue

    end_node = nearest_node_in_set(coarse_end_node, target_trip_nodes)
    if end_node is None:
        continue

    try:
        # [2] Calculate walking path
        path = nx.shortest_path(g, source=start_node, target=end_node, weight='length')
        
        # Convert node path to coordinate path
        path_coords = []
        for node_id in path:
            lat = node_y.get(node_id)
            lon = node_x.get(node_id)
            if lat is not None and lon is not None:
                path_coords.append([lat, lon])
        
        # Compute distance
        dist = 0.0
        for u, v in zip(path[:-1], path[1:]):
            data = g.get_edge_data(u, v)
            if not data: continue
            ed = next(iter(data.values()))
            dist += float(ed.get('length', 1.0))

        # [3] Resolve Stop IDs
        # Start stop: (You might need to refine this if you want the specific stop at start_node)
        # For now, we take the stop from the trip that matches the start_node location if possible
        start_stop_id = None
        candidates_start = node_to_stop_ids.get(start_node, [])
        valid_stops_start = trip_to_stop_set.get(start_trip_id, set())
        for candidate in candidates_start:
            if candidate in valid_stops_start:
                start_stop_id = candidate
                break
        
        # End stop: (The fix we discussed)
        end_stop_id = None
        candidates_end = node_to_stop_ids.get(end_node, [])
        valid_stops_end = trip_to_stop_set.get(end_trip_id, set())
        
        for candidate in candidates_end:
            if candidate in valid_stops_end:
                end_stop_id = candidate
                break
        
        if start_stop_id is None or end_stop_id is None:
            continue

        # [4] Retrieve Route IDs, Agency IDs, Route Names, Route Short Names, and Trip Headsigns
        start_route_id = trip_to_route.get(start_trip_id)
        end_route_id = trip_to_route.get(end_trip_id)
        start_agency_id = route_to_agency.get(start_route_id)
        end_agency_id = route_to_agency.get(end_route_id)
        start_route_short_name = route_to_short_name.get(start_route_id)
        end_route_short_name = route_to_short_name.get(end_route_id)
        start_route_name = route_to_name.get(start_route_id)
        end_route_name = route_to_name.get(end_route_id)
        start_trip_headsign = trip_to_headsign.get(start_trip_id)
        end_trip_headsign = trip_to_headsign.get(end_trip_id)

        # [5] Get stop sequences and coordinates
        start_stop_sequence = trip_stop_to_sequence.get((start_trip_id, start_stop_id))
        end_stop_sequence = trip_stop_to_sequence.get((end_trip_id, end_stop_id))
        
        start_stop_coords = stop_to_coords.get(start_stop_id, {})
        end_stop_coords = stop_to_coords.get(end_stop_id, {})

        records.append({
            'start_trip_id': start_trip_id,
            'end_trip_id': end_trip_id,
            'start_route_id': start_route_id,
            'end_route_id': end_route_id,
            'start_agency_id': start_agency_id,
            'end_agency_id': end_agency_id,
            'start_route_short_name': start_route_short_name,
            'end_route_short_name': end_route_short_name,
            'start_route_name': start_route_name,
            'end_route_name': end_route_name,
            'start_trip_headsign': start_trip_headsign,
            'end_trip_headsign': end_trip_headsign,
            'start_stop_id': start_stop_id,
            'end_stop_id': end_stop_id,
            'start_stop_sequence': start_stop_sequence,
            'end_stop_sequence': end_stop_sequence,
            'start_stop_lat': start_stop_coords.get('stop_lat'),
            'start_stop_lon': start_stop_coords.get('stop_lon'),
            'end_stop_lat': end_stop_coords.get('stop_lat'),
            'end_stop_lon': end_stop_coords.get('stop_lon'),
            'walking_distance_m': dist,
            'walking_path_coords': path_coords,
        })

    except Exception as e:
        failed += 1
print(f"Built {len(records)} pathways. Failed: {failed}")

# Re-create the DataFrame

trip_pathways_df = pd.DataFrame.from_records(records)
trip_pathways_df.head()

Processing 6506 proximity candidates...
Built 6506 pathways. Failed: 0


,start_trip_id,end_trip_id,start_route_id,end_route_id,start_agency_id,end_agency_id,start_route_short_name,end_route_short_name,start_route_name,end_route_name,...,start_stop_id,end_stop_id,start_stop_sequence,end_stop_sequence,start_stop_lat,start_stop_lon,end_stop_lat,end_stop_lon,walking_distance_m,walking_path_coords
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,3SynmEGmJSDoBYRTRgEkj-07:00:00,lgA0Qn219kk386RQHCTQ7,rJNlVBeJYLnin9iUCfQ8_,P_O_14,P_O_14,Microbus,Microbus,Bakus - El-Awayed,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),...,310,310,2,1,31.223723,29.975230,31.223723,29.975230,0.000000,"[[31.2237299, 29.9752465]]"
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,mOAyR9hC46n6ORhDBT2l7-07:00:00,lgA0Qn219kk386RQHCTQ7,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,Bakus - El-Awayed,Bakus - El-Awayed,...,443,443,3,1,31.230842,29.972046,31.230842,29.972046,0.000000,"[[31.230715, 29.9721482]]"
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,xg6X_6yI3-x6agQaIKy0r,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,Abo Soliman - El-Mansheya,Bakus - El-Awayed,...,308,309,12,1,31.224578,29.979563,31.225227,29.980052,39.742259,"[[31.2252996, 29.9797375], [31.2252586, 29.979..."
3,_npHlyCCY7o0R20RyqvT8-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,rJNlVBeJYLnin9iUCfQ8_,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),Bakus - El-Awayed,...,311,310,12,2,31.222990,29.973909,31.223723,29.975230,7073.199280,"[[31.2229663, 29.9734203], [31.2042148, 29.954..."
4,-vGMBy-ffWWJVczaRlv5z-07:00:00,8wpT6YlGJmfsskCDg8AJT-07:00:00,d1tk5YD606wPnGF4CLm4i,RSdPdtwiGzlvedMSUpA36,P_O_14,Bus,Microbus,Bus 243,El-Mawqaf El-Geded - Sidi Gabir,El-Awayed - Sidi Gabir,...,321,322,6,1,31.214939,29.945402,31.214245,29.946088,139.949724,"[[31.2149388, 29.9454022], [31.214973, 29.9458..."


## 7. Save and Preview Results

This cell saves the pathway results to CSV and displays a preview of the generated pathways data.

In [6]:
# Save to CSV and preview
output_csv = Path("trip_pathways.csv")
trip_pathways_df.to_csv(output_csv, index=False)
print(f"Saved trip pathways to {output_csv.resolve()}")
trip_pathways_df.head(5)


Saved trip pathways to C:\Users\ahmed\Downloads\my-demo\api\routing_module\data\trip_pathways.csv


,start_trip_id,end_trip_id,start_route_id,end_route_id,start_agency_id,end_agency_id,start_route_short_name,end_route_short_name,start_route_name,end_route_name,...,start_stop_id,end_stop_id,start_stop_sequence,end_stop_sequence,start_stop_lat,start_stop_lon,end_stop_lat,end_stop_lon,walking_distance_m,walking_path_coords
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,3SynmEGmJSDoBYRTRgEkj-07:00:00,lgA0Qn219kk386RQHCTQ7,rJNlVBeJYLnin9iUCfQ8_,P_O_14,P_O_14,Microbus,Microbus,Bakus - El-Awayed,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),...,310,310,2,1,31.223723,29.975230,31.223723,29.975230,0.000000,"[[31.2237299, 29.9752465]]"
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,mOAyR9hC46n6ORhDBT2l7-07:00:00,lgA0Qn219kk386RQHCTQ7,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,Bakus - El-Awayed,Bakus - El-Awayed,...,443,443,3,1,31.230842,29.972046,31.230842,29.972046,0.000000,"[[31.230715, 29.9721482]]"
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,xg6X_6yI3-x6agQaIKy0r,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,Abo Soliman - El-Mansheya,Bakus - El-Awayed,...,308,309,12,1,31.224578,29.979563,31.225227,29.980052,39.742259,"[[31.2252996, 29.9797375], [31.2252586, 29.979..."
3,_npHlyCCY7o0R20RyqvT8-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,rJNlVBeJYLnin9iUCfQ8_,lgA0Qn219kk386RQHCTQ7,P_O_14,P_O_14,Microbus,Microbus,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),Bakus - El-Awayed,...,311,310,12,2,31.222990,29.973909,31.223723,29.975230,7073.199280,"[[31.2229663, 29.9734203], [31.2042148, 29.954..."
4,-vGMBy-ffWWJVczaRlv5z-07:00:00,8wpT6YlGJmfsskCDg8AJT-07:00:00,d1tk5YD606wPnGF4CLm4i,RSdPdtwiGzlvedMSUpA36,P_O_14,Bus,Microbus,Bus 243,El-Mawqaf El-Geded - Sidi Gabir,El-Awayed - Sidi Gabir,...,321,322,6,1,31.214939,29.945402,31.214245,29.946088,139.949724,"[[31.2149388, 29.9454022], [31.214973, 29.9458..."


## 8. Add Route Names for Better Readability

This cell enhances the pathway data by adding human-readable route names (from route_long_name) to make the results easier to interpret.

In [7]:
# Show some examples
trip_pathways_df[['start_trip_id', 'start_route_short_name', 'start_route_name', 'start_trip_headsign', 'start_agency_id', 
                  'end_trip_id', 'end_route_short_name', 'end_route_name', 'end_trip_headsign', 'end_agency_id',
                  'start_stop_id', 'start_stop_sequence', 'end_stop_id', 'end_stop_sequence',
                  'start_stop_lat', 'start_stop_lon', 'end_stop_lat', 'end_stop_lon',
                  'walking_distance_m']].head(3)

,start_trip_id,start_route_short_name,start_route_name,start_trip_headsign,start_agency_id,end_trip_id,end_route_short_name,end_route_name,end_trip_headsign,end_agency_id,start_stop_id,start_stop_sequence,end_stop_id,end_stop_sequence,start_stop_lat,start_stop_lon,end_stop_lat,end_stop_lon,walking_distance_m
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Microbus,Bakus - El-Awayed,Bakus,P_O_14,3SynmEGmJSDoBYRTRgEkj-07:00:00,Microbus,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),El-Mansheya,P_O_14,310,2,310,1,31.223723,29.975230,31.223723,29.975230,0.000000
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Microbus,Bakus - El-Awayed,Bakus,P_O_14,mOAyR9hC46n6ORhDBT2l7-07:00:00,Microbus,Bakus - El-Awayed,El-Awayed,P_O_14,443,3,443,1,31.230842,29.972046,31.230842,29.972046,0.000000
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,Microbus,Abo Soliman - El-Mansheya,Abo Soliman,P_O_14,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Microbus,Bakus - El-Awayed,Bakus,P_O_14,308,12,309,1,31.224578,29.979563,31.225227,29.980052,39.742259


## 9. Visualization Functions

This cell defines functions for visualizing trip pairs and their walking pathways on interactive Folium maps:
- `plot_trip_pair_with_path()`: Creates maps showing two trip routes and the walking connection between them
- Uses GTFS shapes for accurate trip route visualization
- Displays walking paths with start/end markers

In [8]:
# Visualization: plot two trips and the walking path between them
import folium

def _get_trip_shape_coords(trip_id):
    """Get trip shape coordinates from GTFS shapes data"""
    shape_id = trip_to_shape.get(trip_id)
    if shape_id and shape_id in shape_to_points:
        return shape_to_points[shape_id]
    return []

def plot_trip_pair_with_path(start_trip_id, end_trip_id, save_html=None, map_tiles="cartodbpositron"):
    """
    Plot two trips and the walking path between them using GTFS shapes and pathways data.
    """
    # Find pathway row
    row = trip_pathways_df[(trip_pathways_df['start_trip_id'] == start_trip_id) & 
                          (trip_pathways_df['end_trip_id'] == end_trip_id)]
    if row.empty:
        raise ValueError("No pathway found for the given trip pair in trip_pathways_df")
    row = row.iloc[0]
    walking_coords = row['walking_path_coords']

    # Get trip shapes
    t1_coords = _get_trip_shape_coords(start_trip_id)
    t2_coords = _get_trip_shape_coords(end_trip_id)
    
    # Calculate map center from all available coordinates
    all_coords = []
    if t1_coords: all_coords.extend(t1_coords)
    if t2_coords: all_coords.extend(t2_coords)
    if walking_coords: all_coords.extend(walking_coords)
    
    if all_coords:
        center_lat = sum(coord[0] for coord in all_coords) / len(all_coords)
        center_lon = sum(coord[1] for coord in all_coords) / len(all_coords)
    else:
        center_lat, center_lon = 31.2, 29.9  # Default Alexandria center

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    # Plot trip shapes
    if t1_coords:
        folium.PolyLine(t1_coords, color="#1f77b4", weight=5, opacity=0.8, 
                       tooltip=f"Trip {start_trip_id}").add_to(m)
    if t2_coords:
        folium.PolyLine(t2_coords, color="#ff7f0e", weight=5, opacity=0.8, 
                       tooltip=f"Trip {end_trip_id}").add_to(m)

    # Plot walking path
    if walking_coords:
        folium.PolyLine(walking_coords, color="#2ca02c", weight=6, opacity=0.9, 
                       tooltip=f"Walk {len(walking_coords)} pts").add_to(m)
        # Mark start/end of walking path
        folium.CircleMarker(walking_coords[0], radius=5, color="#2ca02c", fill=True, 
                           tooltip="Walk start").add_to(m)
        folium.CircleMarker(walking_coords[-1], radius=5, color="#2ca02c", fill=True, 
                           tooltip="Walk end").add_to(m)

    if save_html:
        m.save(save_html)
    return m

# Example usage:
# plot_trip_pair_with_path('trip1_id', 'trip2_id', save_html='trip_pair_map.html')


## 10. Query Specific Trip Pathways

This cell demonstrates how to filter and examine pathways for a specific trip ID.

In [9]:
trip_pathways_df[trip_pathways_df['start_trip_id'] == 'oL-cc2-4cc5VXM7r-ShUc-07:00:00'].head(3)

,start_trip_id,end_trip_id,start_route_id,end_route_id,start_agency_id,end_agency_id,start_route_short_name,end_route_short_name,start_route_name,end_route_name,...,start_stop_id,end_stop_id,start_stop_sequence,end_stop_sequence,start_stop_lat,start_stop_lon,end_stop_lat,end_stop_lon,walking_distance_m,walking_path_coords
110,oL-cc2-4cc5VXM7r-ShUc-07:00:00,0h48FVJYF7q0a-Fr5KoJ2-07:00:00,ujMXATExhhRj9qgxcopKv,2sefONCZ5v5w4bCF1ybYX,P_O_14,Minibus,Microbus,Minibus 20,Street 45 - Train Station (El-Shohada Square),Al-Milaha - Train Station (El-Shohada Square),...,402,403,15,1,31.196257,29.91283,31.197209,29.911435,200.395236,"[[31.196281, 29.9129041], [31.1961716, 29.9125..."
276,oL-cc2-4cc5VXM7r-ShUc-07:00:00,3JTmhQmnxSU9cPkJSaSlE-07:00:00,ujMXATExhhRj9qgxcopKv,LrIznn0Nk3YYvE6SV4NT_,P_O_14,Bus,Microbus,Bus 739,Street 45 - Train Station (El-Shohada Square),Asafra - Train Station (El-Shohada Square),...,402,402,15,29,31.196257,29.91283,31.196257,29.912830,0.000000,"[[31.196281, 29.9129041]]"
495,oL-cc2-4cc5VXM7r-ShUc-07:00:00,4QUtLrt9QqoQBADynOzAM-07:00:00,ujMXATExhhRj9qgxcopKv,SqToUMEw-z1wSe52v0JGh,P_O_14,P_O_14,Microbus,Microbus,Street 45 - Train Station (El-Shohada Square),Abu Qir - Train Station (El-Shohada Square),...,402,402,15,12,31.196257,29.91283,31.196257,29.912830,0.000000,"[[31.196281, 29.9129041]]"


## 11. Visualize Example Trip Connection

This cell creates an interactive map visualization showing the walking pathway between two specific trips.

In [10]:
plot_trip_pair_with_path('oL-cc2-4cc5VXM7r-ShUc-07:00:00', 'aL3IoFC8Ob46ci80-r5J_-07:00:00')